In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!uv add -q transformers datasets peft bitsandbytes accelerate

In [ ]:
MODEL="HuggingFaceTB/SmolLM-135M"
DATASET="flytech/python-codes-25k"
DATA_COLUMN="output"

SEQ_LEN=512

MAX_STEPS=2000
BATCH_SIZE=16
GR_ACC_STEPS=1
LR=5e-4
LR_SCHEDULER_TYPE="cosine"
WEIGHT_DECAY=0.01
NUM_WARMUP_STEPS=100
EVAL_FREQ=100
SAVE_FREQ=100
LOG_FREQ=20
OUTPUT_DIR="./output"
BF16=True
FP16=False

FIM_RATE=0.0
FIM_SPM_RATE=0.0

#LORA
LORA_R=8
LORA_ALPHA=32
LORA_DROPOUT=0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
USE_NESTED_QUANT=True
BNB_4BIT_COMPUTE_TYPE="bfloat16"
SEED=0

In [32]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    logging,
    set_seed,
    BitsAndBytesConfig,
)

set_seed(SEED)

In [33]:
from datasets import load_dataset
import torch
from tqdm import tqdm

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(example):
    text = example[DATA_COLUMN]
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=SEQ_LEN,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [52]:
dataset = load_dataset(
    DATASET,
    split="train",
)
dataset = dataset.shuffle(seed=SEED)

valid_data = dataset.select(range(3000))
train_data = dataset.select(range(3000), len(dataset))
train_data = train_data.map(preprocess, remove_columns=train_data.column_names)
valid_data = valid_data.map(preprocess, remove_columns=valid_data.column_names)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [62]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

def chars_token_ratio(dataset, tokenizer, data_column, nb_examples=400):
    total_char, total_tokens = 0, 0
    for i, example in enumerate(dataset):
        if i >= nb_examples:
            break
        text = example[data_column]
        # Convert to string just in case
        if not isinstance(text, str):
            text = str(text)
        total_char += len(text)
        total_tokens += len(tokenizer(text).tokens())
    return total_char / total_tokens
DATA_COLUMN = "output" if "output" in train_data.column_names else train_data.column_names[0]
chars_per_token = chars_token_ratio(train_data, tokenizer, DATA_COLUMN)
print(f"The character to token ratio of the dataset is: {chars_per_token:.2f}")

The character to token ratio of the dataset is: 1.00


In [54]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.tuners.lora import LoraLayer

In [55]:
compute_dtype = getattr(torch, BNB_4BIT_COMPUTE_TYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=USE_NESTED_QUANT
)

device_map ="auto"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True,
    use_cache=False,
    # attn_implementation="flash_attention_2" #not for cpu, when moving to cuda enable it, ususally linux+cuda
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [56]:
model = prepare_model_for_kbit_training(model)

In [57]:
peft_config = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805


In [58]:
print(f"lora target modules: {LORA_TARGET_MODULES}")

lora target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']


In [59]:
train_data.start_iteration = 0

training_args = TrainingArguments(
    output_dir=f"mitishraina/{OUTPUT_DIR}",
    dataloader_drop_last=True,
    # evaluation_strategy="steps",
    save_strategy="steps",
    max_steps=MAX_STEPS,
    eval_steps=EVAL_FREQ,
    save_steps=SAVE_FREQ,
    logging_steps=LOG_FREQ,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=NUM_WARMUP_STEPS,
    gradient_accumulation_steps=GR_ACC_STEPS,
    gradient_checkpointing=True,
    fp16=FP16,
    bf16=BF16,
    weight_decay=WEIGHT_DECAY,
    push_to_hub=True,
    # include_tokens_per_second=True,
)

In [60]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
)
print("Training...")
trainer.train()

Training...


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
trainer.push_to_hub()